# Step 2: AWQ 量化（W4A16，llm-compressor 统一路径）

**目标**：用 `llmcompressor` 的 `AWQModifier` + `QuantizationModifier` 把 Qwen2.5-7B-Instruct 量化成 **W4A16（4-bit 权重 / 16-bit 激活）**——显存省到约 1/4，激活保持高精度（所以对长上下文 / 注意力类负载友好），产物同样是 compressed-tensors 格式。

**对应 OUTLINE 课时**：2.4 AWQ W4A16 全流程（~55 分钟）。

> AWQ（Activation-aware Weight Quantization）的核心思想：**不是所有权重都同等重要**——保护那些"大激活输入对应"的显著权重通道（通过 per-channel scale 把它们的量化误差缩小），就能在 4-bit 下几乎不掉点。与 FP8/SmoothQuant 不同，AWQ 只压权重、不压激活（W4A16）。

> **先看 s0**：本 lab 走的就是 s0 讲的那条 `oneshot(model, dataset, recipe)` 通路。AWQ 在通路里的位置 = **两段式 recipe**：第一段 `AWQModifier` 搜显著通道 scale，第二段 `QuantizationModifier(W4A16_ASYM)` 压 INT4；激活保持 FP16（weight-only）。本 lab 目标：**端到端跑通 + 理解每步为何这么干**。

In [ ]:
%%capture
import subprocess, pathlib, json
import torch
import ipytest
ipytest.autoconfig()
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.transform.awq import AWQModifier

In [ ]:
# Setup cell（三 step 共用同一套模块根解析；规范见 course/NOTEBOOK_CONVENTIONS.md 第 2 节）。
# 每个 notebook 自包含地向上发现模块根（含 steps/ + pyproject.toml），绝不依赖裸相对路径或仓库根。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml 的目录）；请在模块目录内启动 jupyter")

MODULE_ROOT = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 scripts/download_model.sh 一致
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"     # L3 先在 0.5B 上验，再上 7B
OUT_ROOT       = MODULE_ROOT / "out"                                 # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)
print("GPU OK" if __import__("torch").cuda.is_available() else "无 GPU（仅 L1/L2 可跑）")


## 原理：AWQ 的两段式 recipe 与「保护显著通道」直觉

llmcompressor 的 AWQ 是**两段式 recipe**（与 SmoothQuant 的两段式结构对称）：

- **第一段** `AWQModifier()`：在校准数据上前向，为每个 Linear **搜出一组 per-channel scale**——这是 AWQ 的核心，下面详述。
- **第二段** `QuantizationModifier(scheme="W4A16_ASYM")`：把权重真正压成 4-bit（第一段算出的 scale 在这里生效），激活保持 FP16 不量化。

> 填空时你会自己拼出这两段（不给现成 recipe 代码——那是你要写的）。第二段的 targets / scheme / ignore 三参数，第一段 AWQModifier 用默认值即可。

### AWQ 核心直觉：为什么「大激活 → 保护对应权重」

INT4 权重量化是**均匀网格**（16 个离散值）。一个权重通道的量化误差 ≈ `网格间距 / 该通道权重的量程`。所以**量程大的通道相对误差小、量程小的通道相对误差大**。AWQ 的洞察（与 SmoothQuant 的「把难从激活搬到权重」是同一类数学等价变换思路，但方向不同）：

1. **挑出「显著通道」**：在校准数据上前向，统计每个输入通道的激活幅值 `s_j ∝ mean(|X_j|)`。激活大的通道，其对应权重 `W[:, j]` 的微小量化误差会被大激活放大，最终对输出影响最大——这些就是要保护的「显著通道」。
2. **给显著通道的权重乘 scale（放大它）**：`W'[:, j] = W[:, j] · s_j`。权重被放大后，量程变大，**同样的 INT4 网格下相对量化误差变小**（误差 ≈ 网格间距/量程，量程 ↑ 则误差 ↓）。
3. **给对应激活除 scale（补偿，保证前向不变）**：`X'_j = X_j / s_j`。于是 `X'_j · W'[:, j] = (X_j/s_j)·(W[:, j]·s_j) = X_j·W[:, j]`——数学等价，前向输出分毫不变。
4. 真正量化的是 `W'`（已放大、好量化）；推理时加载 `W'` 的 int4 + 这组 scale，激活 `X` 在线除以 scale 还原。**净效果：显著通道的 4-bit 误差被压小，非显著通道本就不重要——整体掉点极小。**

一句话因果链：**大激活 → 放大其对应权重 → INT4 相对误差变小 → 精度保住**。`AWQModifier()` 就是自动完成 1–3 步的网格搜索（在预设的 scale 候选里挑使量化误差最小的 s_j），你不用手写搜索逻辑。

> 对比 SmoothQuant（Step 3）：SmoothQuant 把「难」从激活搬到权重（给激活除 s、权重乘 s，治激活离群点）；AWQ 只压权重、不动激活，靠放大显著权重通道来降低其量化误差。两者都用「乘 scale + 除 scale」的等价变换，但目标不同：SmoothQuant 为激活可量化，AWQ 为权重少掉点。

`QuantizationModifier(scheme="W4A16_ASYM")` 细节：权重 4-bit **非对称**（ASYM，带 zero-point）、**group-wise**（默认 group_size=128，每 128 个权重共享一组 scale/zp）；激活**不量化**（A16）。**需要校准数据**（AWQ 要看激活分布挑显著通道）：用 `wikitext-2`，AWQ 极省样本，128–256 条即可（OUTLINE 2.2）。

**易错点（OUTLINE 标注）**：
- 旧 `from llmcompressor.modifiers.awq import AWQModifier` 已**废弃**（兼容 shim），当前路径是 `llmcompressor.modifiers.transform.awq`。
- AutoAWQ 的 `quant_config` **没有 `exclude_modules` 字段**——AWQ 排除层走加载侧（`AwqConfig.modules_to_not_convert`，默认 `['lm_head']`），量化期 AWQ 硬编码跳过 lm_head。课程用 llmcompressor 统一路径，靠 `ignore=["lm_head"]` 在量化期显式排除。


### 校准数据坑：datasets 命名空间（必读，否则 `HfUriError`）

校准数据本步用 `wikitext-2`。**新版 `datasets` 要求用带命名空间的 repo id**：

- ✅ `load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")`
- ❌ `load_dataset("wikitext", ...)` —— 老博客常这么写，新版会抛 `HfUriError`（找不到裸 `wikitext` 这个 repo）。

`Salesforce/wikitext` 是 AWQ 官方实现用的同一份数据；`wikitext-2-raw-v1` 是配置名（raw 表示未做 tokenize 的纯文本）。本步填空 `build_calibration_dataset` 里走网络分支时就要用到这个命名空间——免网络分支（传 `raw_texts`）则不触发，便于在无网/tiny 环境验证。

## 端到端：AWQ 在通路上每步为什么这么干

套用 s0 的通路（`model + dataset + recipe → oneshot → 产物`），AWQ 每步的 why：

**① dataset 步——为什么 AWQ 要校准数据？**
AWQ 要挑「显著通道」（大激活对应的权重通道），**必须看真实激活幅值**才能挑——所以需要校准数据跑前向。用 `wikitext-2`（`Salesforce/wikitext` 命名空间），AWQ 极省样本，128–256 条就够（不像 GPTQ 要 512+）。

**② recipe 步——为什么是两段？**
- 第一段 `AWQModifier()`：在校准数据上前向，为每个 Linear **搜 per-channel scale**（把显著权重通道放大，使其在 INT4 网格下相对误差变小）。这一段**不带参数**（网格搜索用默认候选值）。
- 第二段 `QuantizationModifier(scheme="W4A16_ASYM", targets="Linear", ignore=["lm_head"])`：把权重**真正压成 4-bit**，第一段算的 scale 在这里生效。三个参数各有 why：
  - `targets="Linear"`：只量化 Linear 层（embedding/norm 等保持 FP）。
  - `scheme="W4A16_ASYM"`：4-bit 非对称（带 zero-point，适合权重分布）、group-wise（每 128 个权重共享一组 scale/zp）；激活不量化（A16）。
  - `ignore=["lm_head"]`：lm_head 对输出 logits 最敏感、量化易掉点，跳过。

**③ oneshot 步——调用时内部发生什么？**
回扣 s0 通路：`oneshot(model, dataset, recipe)` → 用校准数据前向收集激活 → 按 recipe 顺序：先 AWQModifier 搜 scale、再 QuantizationModifier 压 INT4 → 保存。你填的 recipe 决定了它怎么量化。

**④ 产物步——怎么验证量化对了？**
读产物 `config.json` 的 `quantization_config`，确认 W4A16 证据：`weights.num_bits=4` / `weights.symmetric=False` / `weights.group_size=128` / `weights.input_activations=None`（激活没量化）。这就是下面 `awq_config_summary` 要抽的字段。

## 本步填空

你要实现三个函数，搭出 AWQ 量化能力：

1. **`build_awq_recipe(ignore, scheme)`** —— 构造 AWQ 两段式 recipe：第一段 `AWQModifier`（搜显著权重 scale），第二段 `QuantizationModifier`（targets / scheme / ignore 三参数）。
2. **`build_calibration_dataset(tokenizer, n_samples, seq_len, raw_texts=None)`** —— 构造校准数据集（返回 `datasets.Dataset`，含 `"text"` 列的**原始文本**——oneshot 内部会 tokenize，你这里只负责取样本）。`raw_texts` 非 None 时直接包装（免网络、用于 tiny/测试）；为 None 时从 `wikitext-2` 加载、shuffle、取前 `n_samples` 条非空文本。
3. **`awq_config_summary(qc)`** —— 从产物 `quantization_config` 抽出 W4A16 关键字段（num_bits=4 / symmetric=False / group_size / 无 input_activations）。

> **注意签名里没有 `group_size`**：`build_awq_recipe(ignore, scheme)` 只有这两个参数。W4A16_ASYM 的 group_size=128 是 scheme 内置默认，`QuantizationModifier` 本身也不接收 group_size 关键字（显式传会 `ValidationError`）。所以你不需要、也不能在这里传 group_size。

填完每个函数后跑紧随其后的 `%%ipytest` cell 验证（L1）。

In [ ]:
def build_awq_recipe(ignore=("lm_head",), scheme="W4A16_ASYM"):
    """返回 AWQ 两段式 recipe（list of modifiers）。

    **为什么这么设计（填前先想）**：这个函数产出 s0 通路里的 `recipe`——一个 modifier 有序列表。AWQ 要两段：第一段搜 scale（不带参数，回顾端到端第②步）、第二段压 INT4（要 targets/scheme/ignore，各参数 why 见端到端第②步）。scheme 用入参（别写死，便于复用）；ignore 入参是 tuple 要转 list（QuantizationModifier 要求 list）。

    返回 [第一段, 第二段] 两个 modifier 组成的 list：
      - 第一段：负责 AWQ 的核心——搜每个 Linear 的显著权重 scale（"平滑"显著通道）。
        想想：这一段需要你给它什么参数吗？（回顾原理段里它的作用）。
      - 第二段：负责把权重真正压成 4-bit。用通用 QuantizationModifier，
        它需要三个关键参数——targets（量化谁）、scheme（量化方案，用入参 scheme）、
        ignore（跳过谁）。注意 scheme 已作为入参传入，别写死。

    易错点（真坑，别踩）：
      - QuantizationModifier **没有** group_size 关键字参数；W4A16_ASYM 的
        group_size=128 是 scheme 内置默认。显式传 group_size=... 会 ValidationError。
      - ignore 必须是 list；入参 ignore 是 tuple，需转换。
    """
    # TODO: 构造两个 modifier 并返回它们的 list。
    #   提示方向（不给具体实参）：
    #     1) 第一段 modifier 不带参数即可（它的网格搜索用默认值）。
    #     2) 第二段是 QuantizationModifier，传 targets / scheme / ignore 三个参数——
    #        scheme 直接用入参 scheme；ignore 记得转成 list。
    raise NotImplementedError


# 脚手架（提供）：真正跑 AWQ 的 execution，调用你填好的 recipe + 校准数据
def run_awq_quantize(model, tokenizer, calib_texts, save_dir, n_samples=256, seq_len=2048):
    recipe = build_awq_recipe()
    oneshot(
        model=model, tokenizer=tokenizer,
        dataset=build_calibration_dataset(tokenizer, n_samples=n_samples, seq_len=seq_len,
                                          raw_texts=calib_texts),
        recipe=recipe, max_seq_length=seq_len, num_calibration_samples=n_samples,
    )
    save_dir = pathlib.Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir)
    return save_dir


In [ ]:
def build_calibration_dataset(tokenizer, n_samples=256, seq_len=2048, raw_texts=None):
    """构造 AWQ 校准数据集（datasets.Dataset，列名 "text"）。

    **为什么这么设计（填前先想）**：这个函数产出 s0 通路里的 `dataset`——oneshot 要校准数据看激活分布（端到端第①步）。返回 HF `Dataset`、含 `text` 列（oneshot 内部 tokenize，你只给原始文本）。`raw_texts` 非 None 时直接包装（免网络、用于 tiny/测试）；为 None 时从 `wikitext-2` 加载（命名空间 `Salesforce/wikitext`，老裸 `wikitext` 会 HfUriError）。

    - 若 raw_texts 不为 None：直接包装成 Dataset({"text": [...]})（用于 tiny / 测试，免网络）。
    - 若 raw_texts 为 None：从 wikitext-2 加载（注意新版 datasets 要用命名空间 repo id
      `Salesforce/wikitext` + 配置名 `wikitext-2-raw-v1`，旧的裸 `wikitext` 会 HfUriError），
      shuffle(seed=42) 后取前 n_samples 条**非空**文本。

    返回：datasets.Dataset（含 "text" 列）。oneshot 内部会用 tokenizer 对 "text" 列
    做 tokenize + 按 max_seq_length 截断，所以这里返回原始文本列即可。

    注：签名保留 `tokenizer` 参数是为了与本模块 s2/s3 其它校准数据函数（如 run_awq_quantize
    传入的 tokenizer）签名对齐，但**本实现不使用 tokenizer**——上面 raw_texts 分支直接包装文本、
    wikitext 分支由 oneshot 内部独立加载并 tokenize，故此参数在本函数体内未引用（保留仅为签名一致）。
    """
    # TODO: 实现（提示：from datasets import load_dataset, Dataset）
    raise NotImplementedError

In [ ]:
def awq_config_summary(quantization_config):
    """从 W4A16 产物的 quantization_config 抽出关键字段。

    **为什么这么设计（填前先想）**：这个函数读 s0 通路里的产物 `quantization_config`（端到端第④步），抽出证明「这是 W4A16」的字段：num_bits=4 / symmetric=False（非对称）/ group_size=128 / input_activations=None（激活没量化）。

    返回 dict，至少含：
      - "quant_method"      : str
      - "weights_num_bits"  : int   （4）
      - "weights_symmetric" : bool  （W4A16_ASYM -> False）
      - "weights_group_size": int   （128）
      - "weights_strategy"  : str   （"group"）
      - "has_input_activations": bool  （AWQ 不量化激活 -> False / None）
      - "targets"           : list
    """
    # TODO: 解析并返回
    raise NotImplementedError

In [ ]:
%%ipytest -qq

def test_build_awq_recipe_two_stages():
    recipe = build_awq_recipe()
    assert isinstance(recipe, list) and len(recipe) == 2
    # 第一段是 AWQModifier
    from llmcompressor.modifiers.transform.awq import AWQModifier
    assert isinstance(recipe[0], AWQModifier)
    # 第二段是 QuantizationModifier，scheme=W4A16_ASYM，ignore 含 lm_head
    assert recipe[1].scheme == "W4A16_ASYM"
    assert "lm_head" in list(recipe[1].ignore)
    assert isinstance(recipe[1].ignore, list)

def test_build_awq_recipe_custom_ignore_and_scheme():
    recipe = build_awq_recipe(ignore=("lm_head", "re:visual"), scheme="W4A16_ASYM")
    assert recipe[1].ignore == ["lm_head", "re:visual"]
    assert recipe[1].scheme == "W4A16_ASYM"

def test_build_calibration_dataset_from_raw_texts():
    # 免网络：直接传文本列表
    class FakeTok:
        model_max_length = 99
    texts = ["hello world " * 10, "another sample text", "third one"]
    ds = build_calibration_dataset(FakeTok(), n_samples=2, seq_len=64, raw_texts=texts)
    assert "text" in ds.column_names
    assert len(ds) == 2

def test_awq_config_summary_on_fake_w4a16():
    fake = {
        "quant_method": "compressed-tensors",
        "ignore": ["lm_head"],
        "config_groups": {"group_0": {
            "targets": ["Linear"],
            "weights": {"num_bits": 4, "symmetric": False, "group_size": 128, "strategy": "group"},
            "input_activations": None,
        }},
    }
    s = awq_config_summary(fake)
    assert s["quant_method"] == "compressed-tensors"
    assert s["weights_num_bits"] == 4
    assert s["weights_symmetric"] is False
    assert s["weights_group_size"] == 128
    assert s["weights_strategy"] == "group"
    assert s["has_input_activations"] is False
    assert s["targets"] == ["Linear"]

### L2 在验证什么
tiny 模型上真跑 AWQ 两段式 → 确认通路跑通 + 产物确实是 W4A16（num_bits=4、激活没量化）。这是「通路 + 产物」的缩微验证。

## L2：tiny 模型验证（CPU/GPU 秒~十秒级）

用 tiny Qwen2（hidden 须能被 group_size 128 整除，故 hidden=128）真跑 AWQ 两段式。
注意 AWQ 的 grid search 比 FP8 慢，所以 tiny 用最小规模（2 层、少量样本）。

In [ ]:
from transformers import Qwen2Config, Qwen2ForCausalLM, PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from datasets import Dataset

def make_tiny_tokenizer(vocab_size=512):
    """构造词表对齐 tiny 模型的 word-level tokenizer（避免 token id 越界）。"""
    vocab = {str(i): i for i in range(vocab_size)}
    tk = Tokenizer(WordLevel(vocab=vocab, unk_token="0"))
    tk.pre_tokenizer = Whitespace()
    return PreTrainedTokenizerFast(tokenizer_object=tk, unk_token="0", pad_token="0",
                                   eos_token="0", bos_token="0", model_max_length=64)

def make_tiny_model(vocab_size=512, hidden_size=128):
    cfg = Qwen2Config(num_hidden_layers=2, hidden_size=hidden_size,
                      intermediate_size=hidden_size * 2, num_attention_heads=4,
                      num_key_value_heads=2, vocab_size=vocab_size, tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).eval()

tiny = make_tiny_model()
tiny_tok = make_tiny_tokenizer()
device = "cuda" if torch.cuda.is_available() else "cpu"
tiny.to(device)
print("tiny AWQ 模型就绪:", sum(p.numel() for p in tiny.parameters()), "params")

# 给 tiny 模型一份微型校准文本（词表内 token "0".."49"）
calib_texts = [" ".join(str(i % 50) for i in range(60))] * 8

# 真跑 AWQ（tiny 规模，group search 极小）
tiny_out = OUT_ROOT / "tiny-awq"
run_awq_quantize(tiny, tiny_tok, calib_texts, tiny_out, n_samples=8, seq_len=64)

qc = json.loads((tiny_out / "config.json").read_text())["quantization_config"]
summary = awq_config_summary(qc)
print("AWQ 产物摘要:", summary)
assert summary["weights_num_bits"] == 4
assert summary["weights_symmetric"] is False
assert summary["has_input_activations"] is False
print("L2 PASS：tiny AWQ 流水线跑通，产物为 W4A16")

### L3 在验证什么
真 Qwen2.5（先 0.5B 再 7B）上跑 AWQ → 确认在真实规模也跑通，且权重磁盘占用降到 ~1/4（W4A16 = 4-bit 权重的直接收益）。

## L3：H200 执行（真 Qwen2.5-0.5B 再 7B）

GPU 守卫：无 GPU 自动跳过。有 GPU 则先 0.5B 验证，再 7B 出可部署 W4A16 产物。

In [ ]:
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # 7B 的 tokenizer 同时用于 0.5B 和 7B：0.5B 与 7B 同属 Qwen2.5 族、
    # tokenizer 词表一致（config.architectures 均为 Qwen2ForCausalLM），故可跨模型复用。
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)

    # 先 0.5B 快验
    m05b = AutoModelForCausalLM.from_pretrained(TINY_MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_05b = OUT_ROOT / "qwen05b-awq"
    run_awq_quantize(m05b, tok, None, out_05b, n_samples=128, seq_len=512)
    print("0.5B AWQ done ->", out_05b)
    del m05b; torch.cuda.empty_cache()

    # 再 7B（出可部署产物；AWQ 省 token，256 样本足够）
    m7b = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_7b = OUT_ROOT / "qwen7b-awq"
    run_awq_quantize(m7b, tok, None, out_7b, n_samples=256, seq_len=2048)
    print("7B AWQ done ->", out_7b)
    del m7b; torch.cuda.empty_cache()
else:
    print("跳过：无 GPU（CPU 环境只跑 L1/L2）。")

### 产物检查在验证什么
读 7B AWQ 产物的 `quantization_config`，用 `awq_config_summary` 抽字段 → 确认 W4A16（4-bit/非对称/group=128/激活不量化），并与 FP16（原始）对比显存。这是端到端第④步的落地。

## 产物检查

打印 7B AWQ 产物的 `quantization_config`，并与 FP16 / FP8（若 Step 1 已跑）对比显存。

In [ ]:
def report_awq_artifact(out_dir):
    out_dir = pathlib.Path(out_dir)
    if not out_dir.exists():
        print(f"(跳过：{out_dir} 不存在，可能 L3 未跑)")
        return None
    cfg = json.loads((out_dir / "config.json").read_text())
    qc = cfg["quantization_config"]
    s = awq_config_summary(qc)
    total = sum(f.stat().st_size for f in out_dir.glob("*.safetensors"))
    print(f"== {out_dir.name} ==")
    print(f"  weights: {s['weights_num_bits']}-bit sym={s['weights_symmetric']} "
          f"group={s['weights_group_size']} strategy={s['weights_strategy']}")
    print(f"  input_activations 量化: {s['has_input_activations']}（W4A16 = 不量化激活）")
    print(f"  safetensors 总大小(=权重磁盘占用): {total/1e9:.2f} GB")
    return total

sizes = {"AWQ 7B": report_awq_artifact(OUT_ROOT / "qwen7b-awq")}
fp16 = sum(f.stat().st_size for f in MODEL_DIR.glob("*.safetensors"))
print(f"\n== 原始 FP16 7B(权重磁盘占用): {fp16/1e9:.2f} GB ==")
if sizes["AWQ 7B"]:
    print(f"== AWQ/FP16 权重磁盘比: {sizes['AWQ 7B']/fp16:.2%}（W4A16 理论约 25%~30%）==")
    print("注：以上是*权重磁盘占用*；AWQ int4 权重在推理时可能按需解压，实际峰值显存需 vLLM 实测，见 M4。")
fp8 = OUT_ROOT / "qwen7b-fp8"
if fp8.exists():
    fp8_size = sum(f.stat().st_size for f in fp8.glob("*.safetensors"))
    print(f"== 对比 Step1 FP8: {fp8_size/1e9:.2f} GB ==")